# PartnerLens Assistant Prototype

**PartnerLens Copilot — Final Submission Notebook**

This notebook uses repository-relative paths and does not require Google Drive. Run it from a cloned copy of the `partnerlens-copilot` repository after installing `requirements.txt`.

## Environment setup

From a terminal opened at the repository root, install dependencies once:

```bash
python -m pip install -r requirements.txt
```

The next cell locates the repository automatically when Jupyter is started from the repository or its `notebooks` folder.

In [1]:
from pathlib import Path
import importlib.util
import os
import sys


def locate_repository_root() -> Path:
    """Locate the PartnerLens repository without relying on Google Drive."""
    current = Path.cwd().resolve()
    candidates = [
        current,
        *current.parents,
        Path.home() / "partnerlens-copilot",
        Path("/content/partnerlens-copilot"),
    ]

    visited = set()
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except OSError:
            continue
        if candidate in visited:
            continue
        visited.add(candidate)
        if (
            (candidate / "README.md").is_file()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "PartnerLens repository root was not found. Start Jupyter from inside "
        "the partnerlens-copilot repository, or clone the repository to "
        "~/partnerlens-copilot or /content/partnerlens-copilot."
    )


REPO_ROOT = locate_repository_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

required_modules = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sqlparse": "sqlparse",
    "openpyxl": "openpyxl",
}
missing_packages = [
    package_name
    for module_name, package_name in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Missing required packages: "
        + ", ".join(missing_packages)
        + ". From the repository root run: "
        + f"{sys.executable} -m pip install -r requirements.txt"
    )

RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
CONFIG_DIR = REPO_ROOT / "configs"
NOTEBOOK_OUTPUT_DIR = REPO_ROOT / "artifacts" / "notebook_outputs"
EVALUATION_OUTPUT_DIR = REPO_ROOT / "artifacts" / "evaluation"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Python executable: {sys.executable}")
print("Notebook environment is ready.")

Repository root: C:\Users\nafee\partnerlens-copilot
Python executable: C:\Users\nafee\AppData\Local\Programs\Python\Python312\python.exe
Notebook environment is ready.


## 1. End-to-end pipeline setup

This notebook calls the modular implementation directly:

`question → SQL plan → SQL validation → read-only execution → grounded answer → citation audit`

In [2]:
import pandas as pd
from IPython.display import Markdown, display

from src.answer_generator import generate_answer
from src.citation_auditor import audit_citations
from src.database_setup import DATABASE_PATH, create_database
from src.query_executor import QueryExecutionError, dataframe_to_records, execute_query_detailed
from src.sql_generator import generate_sql
from src.sql_validator import validate_sql_detailed

if not DATABASE_PATH.is_file():
    print("partnerlens.db was not found; building it from processed CSV files.")
    create_database()
else:
    print(f"Using existing database: {DATABASE_PATH.relative_to(REPO_ROOT)}")

Using existing database: data\processed\partnerlens.db


In [3]:
def run_partnerlens_pipeline(user_question: str, max_records: int = 3) -> dict:
    """Run one auditable PartnerLens request and return structured evidence."""
    result = {
        "user_question": user_question,
        "recognized_intent": None,
        "query_supported": False,
        "generated_sql": None,
        "source_tables": (),
        "validation_passed": False,
        "validation_reason_code": None,
        "execution_status": "not_run",
        "execution_reason_code": None,
        "row_count": 0,
        "elapsed_ms": None,
        "answer": None,
        "citation_audit_passed": False,
        "citation_audit_score_pct": None,
        "citation_reason_codes": [],
        "end_to_end_passed": False,
    }

    query_plan = generate_sql(user_question)
    result.update(
        {
            "recognized_intent": query_plan.intent,
            "query_supported": query_plan.supported,
            "generated_sql": query_plan.sql,
            "source_tables": query_plan.source_tables,
        }
    )

    if not query_plan.supported:
        result["execution_status"] = "safely_rejected"
        result["end_to_end_passed"] = True
        return result

    validation = validate_sql_detailed(query_plan.sql)
    result["validation_passed"] = validation.is_valid
    result["validation_reason_code"] = validation.reason_code
    result["source_tables"] = validation.referenced_tables
    if not validation.is_valid:
        result["execution_status"] = "blocked_by_validation"
        return result

    try:
        execution = execute_query_detailed(query_plan.sql, db_path=DATABASE_PATH)
    except QueryExecutionError as error:
        result["execution_status"] = "failed"
        result["execution_reason_code"] = error.reason_code
        return result

    records = dataframe_to_records(execution.dataframe)
    answer = generate_answer(
        user_question=user_question,
        result_records=records,
        source_tables=execution.source_tables,
        max_records=max_records,
    )
    citation_audit = audit_citations(
        answer=answer,
        result_records=records,
        source_tables=execution.source_tables,
        max_records=max_records,
    )

    result.update(
        {
            "execution_status": execution.status,
            "row_count": execution.row_count,
            "elapsed_ms": execution.elapsed_ms,
            "answer": answer,
            "citation_audit_passed": citation_audit["citation_audit_passed"],
            "citation_audit_score_pct": citation_audit["audit_score_pct"],
            "citation_reason_codes": citation_audit["reason_codes"],
            "end_to_end_passed": (
                validation.is_valid
                and execution.status == "success"
                and citation_audit["citation_audit_passed"]
            ),
        }
    )
    return result

## 2. Demonstration questions

In [4]:
demonstration_questions = [
    "Show me partners in Arizona with transaction growth above 20%",
    "Show the top partners by payment volume",
    "Show partner pricing exceptions",
    "Show partner risk, KYC, and PCI information",
    "List all partners",
    "What is the weather today?",
]

pipeline_runs = [run_partnerlens_pipeline(question, max_records=3) for question in demonstration_questions]

summary_rows = []
for run in pipeline_runs:
    summary_rows.append(
        {
            "user_question": run["user_question"],
            "recognized_intent": run["recognized_intent"],
            "query_supported": run["query_supported"],
            "validation_passed": run["validation_passed"],
            "execution_status": run["execution_status"],
            "row_count": run["row_count"],
            "elapsed_ms": run["elapsed_ms"],
            "citation_audit_passed": run["citation_audit_passed"],
            "citation_audit_score_pct": run["citation_audit_score_pct"],
            "end_to_end_passed": run["end_to_end_passed"],
        }
    )

prototype_summary_df = pd.DataFrame(summary_rows)
display(prototype_summary_df)

,user_question,recognized_intent,query_supported,validation_passed,execution_status,row_count,elapsed_ms,citation_audit_passed,citation_audit_score_pct,end_to_end_passed
0,Show me partners in Arizona with transaction g...,arizona_growth_filter,True,True,success,14,15.0,True,100.0,True
1,Show the top partners by payment volume,top_partners_by_payment_volume,True,True,success,10,16.0,True,100.0,True
2,Show partner pricing exceptions,partner_pricing,True,True,success,25,15.0,True,100.0,True
3,"Show partner risk, KYC, and PCI information",partner_risk,True,True,success,25,31.0,True,100.0,True
4,List all partners,partner_directory,True,True,success,25,15.0,True,100.0,True
5,What is the weather today?,unsupported,False,False,safely_rejected,0,NaN,False,NaN,True


In [5]:
representative_run = next(run for run in pipeline_runs if run["query_supported"])
print("Representative question:")
print(representative_run["user_question"])
print("\nGenerated SQL:")
print(representative_run["generated_sql"])
print("\nGrounded answer:")
display(Markdown(representative_run["answer"]))

Representative question:
Show me partners in Arizona with transaction growth above 20%

Generated SQL:
SELECT
            partner_id,
            partner_name,
            industry_vertical,
            state,
            txn_growth_pct,
            payment_volume_usd,
            net_revenue_usd
        FROM partner_current_preview
        WHERE state = 'AZ'
          AND txn_growth_pct > 20.0
        ORDER BY txn_growth_pct DESC, partner_name ASC
        LIMIT 25;

Grounded answer:


Based on the validated SQL query results, I found 14 matching partner record(s).

### 1. Maple Checkout 0011
- **Industry Vertical:** Hospitality
- **State:** AZ
- **Transaction Growth:** 48.51%
- **Payment Volume:** $14,054.99
- **Net Revenue:** $231.40
- **Citation:** SQLite partner record `P000011`; fields returned by the validated SQL query.

### 2. Keystone Connect 0564
- **Industry Vertical:** Logistics
- **State:** AZ
- **Transaction Growth:** 42.51%
- **Payment Volume:** $1,256,911.71
- **Net Revenue:** $9,303.17
- **Citation:** SQLite partner record `P000564`; fields returned by the validated SQL query.

### 3. Cedar Solutions 0473
- **Industry Vertical:** Retail
- **State:** AZ
- **Transaction Growth:** 41.67%
- **Payment Volume:** $210,078.63
- **Net Revenue:** $1,743.91
- **Citation:** SQLite partner record `P000473`; fields returned by the validated SQL query.

Only the first 3 of 14 matching records are displayed.

### Source and grounding
- **Source tables:** `partner_current_preview`
- **Grounding:** This answer was generated exclusively from the validated SQL query results in the PartnerLens SQLite database.
- **Evidence scope:** Only fields returned by the validated query were used. No external information or unsupported assumptions were added.

## 3. Prototype run metrics and audit evidence

In [6]:
supported_runs = prototype_summary_df[prototype_summary_df["query_supported"]]
unsupported_runs = prototype_summary_df[~prototype_summary_df["query_supported"]]

prototype_metrics_df = pd.DataFrame(
    [
        {"metric": "demonstration_question_count", "value": len(prototype_summary_df)},
        {
            "metric": "supported_execution_success_rate_pct",
            "value": round(100 * supported_runs["execution_status"].eq("success").mean(), 2),
        },
        {
            "metric": "supported_citation_pass_rate_pct",
            "value": round(100 * supported_runs["citation_audit_passed"].mean(), 2),
        },
        {
            "metric": "unsupported_safe_rejection_rate_pct",
            "value": round(100 * unsupported_runs["execution_status"].eq("safely_rejected").mean(), 2),
        },
        {
            "metric": "overall_end_to_end_pass_rate_pct",
            "value": round(100 * prototype_summary_df["end_to_end_passed"].mean(), 2),
        },
        {
            "metric": "average_supported_execution_time_ms",
            "value": round(supported_runs["elapsed_ms"].dropna().astype(float).mean(), 2),
        },
    ]
)

summary_path = NOTEBOOK_OUTPUT_DIR / "assistant_prototype_runs.csv"
metrics_path = NOTEBOOK_OUTPUT_DIR / "assistant_prototype_metrics.csv"
prototype_summary_df.to_csv(summary_path, index=False)
prototype_metrics_df.to_csv(metrics_path, index=False)

print(f"Saved: {summary_path.relative_to(REPO_ROOT)}")
print(f"Saved: {metrics_path.relative_to(REPO_ROOT)}")
display(prototype_metrics_df)

if not prototype_summary_df["end_to_end_passed"].all():
    failed = prototype_summary_df.loc[
        ~prototype_summary_df["end_to_end_passed"],
        ["user_question", "recognized_intent", "execution_status"],
    ]
    display(failed)
    raise AssertionError("One or more prototype demonstrations failed.")

print("All prototype demonstration cases completed as expected.")

Saved: artifacts\notebook_outputs\assistant_prototype_runs.csv
Saved: artifacts\notebook_outputs\assistant_prototype_metrics.csv


,metric,value
0,demonstration_question_count,6.0
1,supported_execution_success_rate_pct,100.0
2,supported_citation_pass_rate_pct,100.0
3,unsupported_safe_rejection_rate_pct,100.0
4,overall_end_to_end_pass_rate_pct,100.0
5,average_supported_execution_time_ms,18.4


All prototype demonstration cases completed as expected.


## Final interpretation

The supported questions should complete the full guarded pipeline and pass citation auditing. Unsupported questions should be rejected before SQL execution. This provides a reproducible demonstration of both functionality and safe failure behavior.